In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['svg.fonttype'] = 'none'
from matplotlib.patches import Patch
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import QuantileTransformer, StandardScaler, RobustScaler
import scipy.cluster.hierarchy as sch
from itertools import combinations
import joblib
import warnings
from tabulate import tabulate
import os
%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts
warnings.filterwarnings('ignore')

from scipy.stats import kruskal
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

In [ ]:
data_path = os.path.dirname(os.getcwd()) + '/data'
figure_path = os.path.dirname(os.getcwd()) + '/figures'

In [ ]:
clinical = pd.read_csv('data/ALL_GEX_n=314_10_controls_clinical.csv', index_col=0)
clinical.group.value_counts()

In [ ]:
clinical = clinical[clinical.group.isin(['BCP-ALL'])]

In [ ]:
gex = (pd.read_csv('data/GEX_ALL_n=314_10_controls_corrected_and_normalized.csv',index_col=0).T)  # transpose 
gex[gex.index.isin(clinical.index)].shape

In [ ]:
clinical.index
gex = gex.reindex(clinical.index)
sum(gex.index == clinical.index)

In [ ]:
def create_boxes(text, tablefmt = 'grid'):
    table = [[text]]
    output = tabulate(table, tablefmt=tablefmt)
    print(output)

In [ ]:
annot = pd.read_csv('data/annotateddata.csv', sep = ';')
annot2 = annot[annot.feature == 'gene']
annot2['gene_biotype'] = annot2.gene_biotype.str.strip(';')
annot2.rename(columns = {'gene_name': 'gene'}, inplace = True)
annot2

In [ ]:
# Import results file
results_file = pd.read_excel(data_path + '/results/results.xlsx', sheet_name='Table S15 HeH vs ER-overlap', skiprows=2)
#proteins = list(results_file['Unnamed: 0'])
results_file['gene'] = results_file['Unnamed: 0']
proteins = results_file[['gene']]

annot2[annot2.gene.isin(results_file['gene'])].shape
proteins = proteins.merge(annot2[['gene', 'gene_id']], how = 'left')

In [ ]:
gex = gex.reindex(columns = proteins.gene_id)
gex.columns = proteins.gene
gex_groups = pd.concat([gex, clinical[['Final_Subtype']]],join = 'inner', axis = 1)

In [ ]:
dictcolors = {
    'HeH': '#0F4D92',
    'low HeH': '#1f77b4',
    'iAMP21': '#17becf',
    't(12;21)': '#2ca02c',
    't(12;21)-like': '#98df8a',
    '11q23/MLL': '#FF7F50',
    'NUTM1-r': '#ffbb78',
    'PAX5alt': '#FC8EAC',
    'PAX5 p.Pro80Arg': '#FBCCE7',
    't(1;19)': '#20B2AA',
    'MEF2D-r': '#30D5C8',
    't(9;22)': '#9467bd',
    'ph-like': '#c5b0d5',
    'DUX4-r': '#004953',
    'ZNF384-r': '#8A496B'
}

newnames = [
    'HeH',
    'low HeH',
    'iAMP21',
    'ETV6::RUNX1',
    'ETV6::RUNX1-like',
    'KMT2A-r',
    'NUTM1-r',
    'PAX5alt',
    'PAX5 P80R',
    'TCF3::PBX1',
    'MEF2D-r',
    'BCR::ABL1',
    'BCR::ABL1-like',
    'DUX4-r',
    'ZNF384-r'
]

In [ ]:
name_dict = dict(zip(dictcolors.keys(), newnames))
name_dict.keys()== dictcolors.keys()
newcolors = dict(zip(name_dict.values(), dictcolors.values()))
gex_groups.Final_Subtype.value_counts(dropna = False)
gex_groups['ICC_Subtype'] = gex_groups.Final_Subtype.map(name_dict)
gex_groups.ICC_Subtype.value_counts(dropna = False)

In [ ]:
group_counts = gex_groups['ICC_Subtype'].value_counts()

order = list(newcolors.keys())
palette = list(newcolors.values())

# Define the subtypes to compare
HEH_LABEL = "HeH"
ER_LABELS = {"ETV6::RUNX1", "ETV6::RUNX1-like"}  # ER + ER-like

genes = list(gex_groups.columns[:-2])

# ---------- 1) Compute p-values for all genes (for FDR) ----------
p_kw = []
p_heh = []
p_er = []

for gene in genes:
    # Global Kruskal–Wallis across all subtypes
    groups = [
        gex_groups.loc[gex_groups["ICC_Subtype"] == grp, gene].dropna()
        for grp in order
    ]
    groups_nonempty = [g for g in groups if len(g) > 1]
    if len(groups_nonempty) >= 2:
        _, p = kruskal(*groups_nonempty)
    else:
        p = np.nan
    p_kw.append(p)

    # HeH vs rest (Mann–Whitney U)
    x_heh = gex_groups.loc[gex_groups["ICC_Subtype"] == HEH_LABEL, gene].dropna()
    x_rest = gex_groups.loc[gex_groups["ICC_Subtype"] != HEH_LABEL, gene].dropna()
    if len(x_heh) >= 2 and len(x_rest) >= 2:
        _, p = mannwhitneyu(x_heh, x_rest, alternative="two-sided")
    else:
        p = np.nan
    p_heh.append(p)

    # (ER + ER-like) vs rest (Mann–Whitney U)
    x_er = gex_groups.loc[gex_groups["ICC_Subtype"].isin(ER_LABELS), gene].dropna()
    x_rest2 = gex_groups.loc[~gex_groups["ICC_Subtype"].isin(ER_LABELS), gene].dropna()
    if len(x_er) >= 2 and len(x_rest2) >= 2:
        _, p = mannwhitneyu(x_er, x_rest2, alternative="two-sided")
    else:
        p = np.nan
    p_er.append(p)

# ---------- 2) FDR correction (Benjamini–Hochberg) ----------
def fdr_bh(pvals):
    pvals = np.array(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan)

    mask = ~np.isnan(pvals)
    if mask.sum() > 0:
        _, q, _, _ = multipletests(pvals[mask], method="fdr_bh")
        qvals[mask] = q
    return qvals

q_kw = fdr_bh(p_kw)
q_heh = fdr_bh(p_heh)
q_er = fdr_bh(p_er)

def q_to_threshold(q):
    if not np.isfinite(q):
        return "NA"
    if q < 1e-3:
        return "<0.001"
    if q < 1e-2:
        return "<0.01"
    if q < 5e-2:
        return "<0.05"
    return "ns"

def fmt_sig(q, label):
    return f"{label}: FDR {q_to_threshold(q)}"

# ---------- 3) Plot per gene with annotations ----------
for i, gene in enumerate(genes):

    fig, ax = plt.subplots(1, 1, figsize=(15, 5))

    sns.boxplot(
        data=gex_groups,
        x="ICC_Subtype",
        y=gene,
        order=order,
        palette=palette,
        ax=ax
    )

    labels = [f"{grp}\n(n={group_counts.get(grp, 0)})" for grp in order]
    ax.set_xticklabels(labels, rotation=45, ha="right")

    ax.set_xlabel("")
    ax.set_ylabel("Expression (Log2)")
    ax.set_title(gene)

    # Build text (use q-values; if we prefer raw p-values, swap q_* with p_*)
    def fmt_q(q, label):
        return f"{label}:adj.P-val q={q:.2e}" if np.isfinite(q) else f"{label}: NA"

    text_lines = [
        fmt_sig(q_kw[i], "All subtypes"),
        fmt_sig(q_heh[i], "HeH vs others"),
        fmt_sig(q_er[i], "ER & ER-like vs others"),
        ]

    ax.text(
        0.01, 0.95,
        "\n".join(text_lines),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10
    )

    plt.tight_layout()
    plt.savefig(f"updated_figures/rna_seq_figures/All_leukemias_{gene}.pdf")
    plt.show()


# Simplified plots

In [ ]:
def get_counts(gex_groups, target_label):
    n_target = (gex_groups["ICC_Subtype"] == target_label).sum()
    n_others = (gex_groups["ICC_Subtype"] != target_label).sum()
    return int(n_target), int(n_others)

In [ ]:
# --- Split the 19 biomarkers by the subtype they are upregulated in ---

df_biom = pd.read_excel(
    data_path + '/results/results.xlsx',
    sheet_name='Table S15 HeH vs ER-overlap',
    skiprows=1
)

# Harmonize expected column names if file uses "Unnamed: 1"
if 'Up regulated group' not in df_biom.columns and 'Unnamed: 1' in df_biom.columns:
    df_biom = df_biom.rename(columns={'Unnamed: 1': 'Up regulated group'})
if 'Protein /gene' not in df_biom.columns and 'Unnamed: 0' in df_biom.columns:
    df_biom = df_biom.rename(columns={'Unnamed: 0': 'Protein /gene'})

ER_top_proteins  = df_biom[df_biom['Up regulated group'].astype(str).str.strip() == 'ETV6::RUNX1']['Protein /gene'].astype(str).str.strip().tolist()
HeH_top_proteins = df_biom[df_biom['Up regulated group'].astype(str).str.strip() == 'HeH']['Protein /gene'].astype(str).str.strip().tolist()

print(f"ETV6::RUNX1 proteins: {len(ER_top_proteins)}\n{ER_top_proteins}")
print(f"\nHeH proteins: {len(HeH_top_proteins)}\n{HeH_top_proteins}")

# --- Helper: make long-form dataframe for plotting and stats ---
def make_long_df(gex_groups, gene_list, target_label):
    'Return long dataframe with columns: gene, expression, group2 (target vs others), ICC_Subtype'
    keep_cols = gene_list + ['ICC_Subtype']
    df = gex_groups[keep_cols].copy()
    long = df.melt(id_vars=['ICC_Subtype'], var_name='gene', value_name='expression')
    long['group2'] = np.where(long['ICC_Subtype'] == target_label, target_label, 'Others')
    return long.dropna(subset=['expression'])

def mannwhitney_by_gene(long_df, target_label):
    'Compute p-values target vs Others for each gene in long_df.'
    pvals = {}
    for gene, sub in long_df.groupby('gene'):
        x_t = sub.loc[sub['group2'] == target_label, 'expression'].dropna()
        x_o = sub.loc[sub['group2'] == 'Others', 'expression'].dropna()
        if len(x_t) >= 2 and len(x_o) >= 2:
            _, p = mannwhitneyu(x_t, x_o, alternative='two-sided')
        else:
            p = np.nan
        pvals[gene] = p
    return pvals

def fdr_bh_dict(pvals_dict):
    'BH-FDR on a dict of p-values; returns dict of q-values with same keys.'
    genes = list(pvals_dict.keys())
    pvals = np.array([pvals_dict[g] for g in genes], dtype=float)
    qvals = np.full_like(pvals, np.nan)

    mask = ~np.isnan(pvals)
    if mask.sum() > 0:
        _, q, _, _ = multipletests(pvals[mask], method='fdr_bh')
        qvals[mask] = q

    return dict(zip(genes, qvals))

def q_label(q):
    if not np.isfinite(q):
        return "q = NA"
    if q < 1e-3:
        return "Adj. p-val < 0.001"
    if q < 1e-2:
        return "Adj. p-val < 0.01"
    if q < 5e-2:
        return "Adj. p-val < 0.05"
    return f"Adj. p-val = {q:.2f}"

In [ ]:
def plot_panel(long_df, target_label, gene_list, outpath, n_target, n_others, kind='violin', col_wrap=3):
    pvals = mannwhitney_by_gene(long_df, target_label)
    qvals = fdr_bh_dict(pvals)

    long_df['gene'] = pd.Categorical(long_df['gene'], categories=gene_list, ordered=True)

    if target_label == "HeH":
        target_color = "#FF8C42"
    elif target_label == "ETV6::RUNX1":
        target_color = "#6A4C93"
    else:
        target_color = "#000000"

    palette = {target_label: target_color, "Others": "#D3D3D3"}

    # title label
    if target_label == "ETV6::RUNX1":
        title_label = r'$\mathit{ETV6::RUNX1}$'
        tick_target = rf'$\mathit{{ETV6::RUNX1}}$' + f'\n(n={n_target})'
    else:
        title_label = target_label
        tick_target = f'{target_label}\n(n={n_target})'

    tick_others = f'Others (n={n_others})'

    common_kwargs = dict(
        data=long_df, x='group2', y='expression',
        col='gene', col_wrap=col_wrap,
        order=[target_label, 'Others'],
        height=3.2, aspect=1.0,
        sharey=False,
        palette=palette
    )

    if kind == 'box':
        g = sns.catplot(kind='box', **common_kwargs)
    else:
        g = sns.catplot(kind='violin', inner='box', **common_kwargs)

    # annotate q-values + set tick labels on every facet
    for ax in g.axes.flatten():
        gene = ax.get_title().split(' = ')[-1] if ' = ' in ax.get_title() else ax.get_title()
        q = qvals.get(gene, np.nan)
        ax.set_title(f"{gene} ({q_label(q)})")
        ax.set_xlabel("")
        ax.set_ylabel("Expression (Log2)")

        # enforce tick labels (italic + counts)
        ax.set_xticklabels([tick_target, tick_others])

    g.fig.suptitle(f"{title_label} vs Others", y=1.02, fontsize=14)

    plt.tight_layout()
    os.makedirs(os.path.dirname(outpath), exist_ok=True)
    g.savefig(outpath)
    plt.show()


We compared each subtype to all remaining subtypes combined using a two-sided Mann–Whitney U test, and controlled FDR across the biomarkers within each subtype panel using Benjamini–Hochberg correction

In [ ]:
# ER panel
n_target, n_others = get_counts(gex_groups, ER_TARGET)
er_long = make_long_df(gex_groups, ER_top_proteins, ER_TARGET)
plot_panel(er_long, ER_TARGET, ER_top_proteins,
           outpath="updated_figures/rna_seq_figures/ETV6_RUNX1_vs_others_9genes.pdf",
           n_target=n_target, n_others=n_others,
           kind='violin', col_wrap=5)

# HeH panel
n_target, n_others = get_counts(gex_groups, HEH_TARGET)
heh_long = make_long_df(gex_groups, HeH_top_proteins, HEH_TARGET)
plot_panel(heh_long, HEH_TARGET, HeH_top_proteins,
           outpath="updated_figures/rna_seq_figures/HeH_vs_others_10genes.pdf",
           n_target=n_target, n_others=n_others,
           kind='violin', col_wrap=5)